<!-- STEP A0 -->
# Setup: lab steps A0–A3

Owner: T3. Every part notebook starts with `%run 00_setup.ipynb`; this notebook loads the data,
makes the 60 / 20 / 20 split and defines `report()`. The first line of every cell is a step marker
used by `tools/assemble.py`.

**A0. Libraries.** Colab has pandas, scikit-learn and matplotlib but not the two explanation
libraries. Locally everything comes from `requirements.txt`, so nothing is installed.

In [1]:
# STEP A0
# Install shap and lime only when they are missing (Colab). Locally they come from requirements.txt.
import importlib.util

if importlib.util.find_spec("shap") is None or importlib.util.find_spec("lime") is None:
    %pip install -q shap lime

# scipy 1.18 with scikit-learn 1.6.1 prints a harmless "Unknown solver options: iprint" warning
# every time a logistic regression is trained. It does not change any result, so we hide it.
import warnings
warnings.filterwarnings("ignore", message="Unknown solver options")

<!-- STEP A1 -->
## Step A1: Load the data

Web Page Phishing Detection (Hannousse & Yahiouche, 2021), CC BY 4.0: 11,430 URLs, half phishing
and half legitimate, 87 features; the label is in `status`. See `data/README.md`.

In [2]:
# STEP A1
import os
from pathlib import Path

import numpy as np
import pandas as pd

# The part notebooks live in parts/. Work from the repository root, so that
# data/ and results/ mean the same thing in every notebook.
if Path.cwd().name == "parts":
    os.chdir("..")

DATA = Path("data/dataset_phishing.csv")
if not DATA.exists():
    DATA = Path("dataset_phishing.csv")   # Colab: uploaded with the folder icon

# Every step saves its figures and tables here (TASKLIST section 2.5)
for folder in ["results/figures", "results/tables"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA)                     # the file you downloaded
print(df.shape)                            # should print (11430, 89)
print(df["status"].value_counts())         # 5715 phishing, 5715 legitimate

assert df.shape == (11430, 89), "unexpected dataset shape"
assert (df["status"].value_counts() == 5715).all(), "classes are not 5715 / 5715"
df.head()

(11430, 89)
status
legitimate    5715
phishing      5715
Name: count, dtype: int64


,url,length_url,length_hostname,ip,nb_dots,nb_hyphens,nb_at,nb_qm,nb_and,nb_or,...,domain_in_title,domain_with_copyright,whois_registered_domain,domain_registration_length,domain_age,web_traffic,dns_record,google_index,page_rank,status
0,http://www.crestonwood.com/router.php,37,19,0,3,0,0,0,0,0,...,0,1,0,45,-1,0,1,1,4,legitimate
1,http://shadetreetechnology.com/V4/validation/a...,77,23,1,1,0,0,0,0,0,...,1,0,0,77,5767,0,0,1,2,phishing
2,https://support-appleld.com.secureupdate.duila...,126,50,1,4,1,0,1,2,0,...,1,0,0,14,4004,5828815,0,1,0,phishing
3,http://rgipt.ac.in,18,11,0,2,0,0,0,0,0,...,1,0,0,62,-1,107721,0,0,3,legitimate
4,http://www.iracing.com/tracks/gateway-motorspo...,55,15,0,2,2,0,0,0,0,...,0,1,0,224,8175,8725,0,0,6,legitimate


<!-- STEP A2 -->
## Step A2: Prepare and split

The label becomes 1 = phishing, 0 = legitimate, and the raw `url` text is dropped. We split
60 / 20 / 20: **training** to learn, **validation** to choose settings, **test** only for the final
scores. `stratify` keeps 50% phishing in each part.

In [3]:
# STEP A2
from sklearn.model_selection import train_test_split

# The label: 1 = phishing, 0 = legitimate
y = (df["status"] == "phishing").astype(int)

# The features: everything except the raw URL text and the label.
# .astype(float) avoids type warnings later when we change values.
X = df.drop(columns=["url", "status"]).astype(float)

# 60% train, 20% validation, 20% test, same phishing share in each part
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

print(len(X_train), len(X_val), len(X_test))   # 6858 2286 2286

assert X.shape[1] == 87, "expected 87 features"
assert (len(X_train), len(X_val), len(X_test)) == (6858, 2286, 2286)
for name, part in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{name:10s} phishing share: {part.mean():.3f}")
    assert abs(part.mean() - 0.5) < 0.001, f"{name} is not 50% phishing"

6858 2286 2286
train      phishing share: 0.500
validation phishing share: 0.500
test       phishing share: 0.500


<!-- STEP A3 -->
## Step A3: A helper to score any model

Macro-F1, recall, ROC-AUC and FAR (false-alarm rate: share of legitimate URLs wrongly flagged).
Every model in the lab is scored with this function.

In [4]:
# STEP A3
from sklearn.metrics import (f1_score, recall_score, roc_auc_score,
                             confusion_matrix)

def report(model, X, y):
    """The four numbers we use for every model in this lab (3 decimals)."""
    pred = model.predict(X)
    prob = model.predict_proba(X)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    scores = {
        "macro_F1": f1_score(y, pred, average="macro"),
        "recall":   recall_score(y, pred),     # share of phishing we caught
        "ROC_AUC":  roc_auc_score(y, prob),
        "FAR":      fp / (fp + tn),            # share of legit wrongly flagged
    }
    return {k: round(float(v), 3) for k, v in scores.items()}